In [3]:
import pandas as pd
df = pd.read_csv("../data/milestone1_shruti.csv")

In [4]:
import numpy as np

In [5]:
df.head()

,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,NaN,Approve,Low
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,Early termination fee $300,Approve,Low
2,3,This contract between DriveEasy Finance and Sa...,5.41,24,774,Late fee $50,Reject,High
3,4,This contract between National Motors and Alex...,5.26,48,1028,Late fee $25,Approve,High
4,5,This contract between National Motors and John...,5.97,36,1180,NaN,Approve,Low


In [6]:
df.columns

Index(['contract_id', 'raw_text', 'apr', 'term_months', 'monthly_payment',
       'penalty_clause', 'recommended_action', 'risk_flag'],
      dtype='object')

In [7]:
df = df.rename(columns={"penalty": "penalty_clause"})

In [8]:
df["expected_apr"] = df["apr"]
df["expected_term"] = df["term_months"]
df["expeted_payment"] = df["monthly_payment"]
df["expected_penalty"] = df["penalty_clause"]

In [9]:
df[["penalty_clause","expected_penalty"]].head()

,penalty_clause,expected_penalty
0,NaN,NaN
1,Early termination fee $300,Early termination fee $300
2,Late fee $50,Late fee $50
3,Late fee $25,Late fee $25
4,NaN,NaN


In [10]:
# if extracted APR exactly matches expected
# APR score 1
# if not score 0

# if extracted term months equals expected
# term months score 1
# else score 0

# if extracted penalty text matches expected
# penalty score 1
# else score 0

df["apr_score"] = (df["apr"] == df["expected_apr"]).astype(int)

In [11]:
df["term_score"] = (df["term_months"] == df["expected_term"]).astype(int)

In [12]:
df["penalty_score"] = (df["penalty_clause"] == df["expected_penalty"]).astype(int)

In [13]:
df["total_score"] = (df["apr_score"] + df["term_score"] + df["penalty_score"])

In [14]:
df["quality_percent"] = (df["total_score"]/3)*100
df[["apr_score","term_score","penalty_score","total_score","quality_percent"]].head()

,apr_score,term_score,penalty_score,total_score,quality_percent
0,1,1,0,2,66.666667
1,1,1,1,3,100.000000
2,1,1,1,3,100.000000
3,1,1,1,3,100.000000
4,1,1,0,2,66.666667


In [ ]:
 
df.to_csv("../data/milestone2_evaluationshruti_output.csv", index=False)

In [16]:
def extract_sla_details(row):
 """
 Simulates LLM-based SLA extraction.
 Returns SLA details in JSON format.
 """
 return {
 "monthly_emi": row["monthly_emi"],
 "interest_rate": row["interest_rate"],
 "tenure_months": row["tenure_months"],
 "risk_flag": row["risk_flag"],
 "issue_type": row["issue_type"],
 "recommended_action": row["recommended_action"]
 }


In [18]:
import pandas as pd
contracts_df = pd.read_csv("../data/sample_car_contracts_with_vin.csv")
contracts_df.head()

,id,customer_name,contract_type,vehicle_type,monthly_emi,interest_rate,tenure_months,clause_summary,risk_flag,issue_type,recommended_action,vin
0,1,Rahul Mehta,Car Loan,Sedan,18500,9.5,48,Prepayment allowed only after 24 months with 5...,medium,High prepayment charges,Highlight prepayment penalty to user and sugge...,1HGCM82633A004352
1,2,Anita Rao,Car Lease,SUV,22000,0.0,36,Lessee must pay for all maintenance and insurance,low,Standard maintenance clause,"No action, just explain maintenance responsibi...",1HGCM82633A004352
2,3,James Wilson,Car Loan,Hatchback,14500,11.2,60,Late payment fee of 3% per month on outstandin...,high,Aggressive late fee,Flag clause and suggest user request cap on la...,1HGCM82633A004352
3,4,Meena Iyer,Car Lease,Sedan,21000,0.0,24,"Excess mileage charge of ₹12 per km over 15,00...",medium,High excess mileage rate,Warn user about extra mileage charges and reco...,1HGCM82633A004352
4,5,Arjun Patel,Car Loan,SUV,27500,10.8,72,Floating interest rate linked to lender's inte...,high,Unclear interest benchmark,Explain floating rate risk and suggest asking ...,1HGCM82633A004352


In [19]:
sla_records = []
for _, row in contracts_df.iterrows():
 sla_records.append({
 "contract_id": row["id"],
 "customer_name": row["customer_name"],
 "sla": extract_sla_details(row)
 })
sla_df = pd.DataFrame(sla_records)
sla_df.head()

,contract_id,customer_name,sla
0,1,Rahul Mehta,"{'monthly_emi': 18500, 'interest_rate': 9.5, '..."
1,2,Anita Rao,"{'monthly_emi': 22000, 'interest_rate': 0.0, '..."
2,3,James Wilson,"{'monthly_emi': 14500, 'interest_rate': 11.2, ..."
3,4,Meena Iyer,"{'monthly_emi': 21000, 'interest_rate': 0.0, '..."
4,5,Arjun Patel,"{'monthly_emi': 27500, 'interest_rate': 10.8, ..."


In [20]:
sla_df.to_csv("../data/milestone2_sla_output.csv", index=False)

In [21]:
sla_df.sample(5)

,contract_id,customer_name,sla
6,7,Vikram Shah,"{'monthly_emi': 19500, 'interest_rate': 8.9, '..."
10,11,Pratima Das,"{'monthly_emi': 13200, 'interest_rate': 9.9, '..."
5,6,Sana Khan,"{'monthly_emi': 16500, 'interest_rate': 0.0, '..."
9,10,Kiran Kumar,"{'monthly_emi': 18500, 'interest_rate': 0.0, '..."
3,4,Meena Iyer,"{'monthly_emi': 21000, 'interest_rate': 0.0, '..."


In [22]:
import requests
def get_vehicle_details(vin):
 url =f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValues/{vin}?format=json"
 response = requests.get(url)
 return response.json()["Results"][0]


In [23]:
def extract_vehicle_info(vehicle_raw):
 return {
 "make": vehicle_raw.get("Make"),
 "model": vehicle_raw.get("Model"),
 "year": vehicle_raw.get("ModelYear")
 }


In [24]:
def enrich_contract_with_vehicle(row):
 vehicle_raw = get_vehicle_details(row["vin"])
 vehicle_info = extract_vehicle_info(vehicle_raw)
 return {
 "contract_id": row["id"],
 "customer_name": row["customer_name"],
 "contract_type": row["contract_type"],
 "sla": extract_sla_details(row),
 "vehicle": vehicle_info
 }


In [25]:
combined_records = []
for _, row in contracts_df.iterrows():
 combined_records.append(enrich_contract_with_vehicle(row))
combined_df = pd.DataFrame(combined_records)
combined_df.head()

,contract_id,customer_name,contract_type,sla,vehicle
0,1,Rahul Mehta,Car Loan,"{'monthly_emi': 18500, 'interest_rate': 9.5, '...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
1,2,Anita Rao,Car Lease,"{'monthly_emi': 22000, 'interest_rate': 0.0, '...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
2,3,James Wilson,Car Loan,"{'monthly_emi': 14500, 'interest_rate': 11.2, ...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
3,4,Meena Iyer,Car Lease,"{'monthly_emi': 21000, 'interest_rate': 0.0, '...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."
4,5,Arjun Patel,Car Loan,"{'monthly_emi': 27500, 'interest_rate': 10.8, ...","{'make': 'HONDA', 'model': 'Accord', 'year': '..."


In [26]:
combined_df.to_csv("../data/milestone2_evaluation_output.csv", index=False)